# MLP: KL и убывающий шум в весах

Та же функция и данные; обычная MLP сразу на 128 ReLU-нейронах, все параметры обучаемы. 1600 итераций. Уровень шума убывает по косинусу от 1 до 0,05 на первых 1599 итерациях: `sigma = sigma_before_final + (sigma_start - sigma_before_final) * (1 + cos(pi * progress)) / 2`. **На последней σ=0 и обычный MSE**.

Для `h(x)=ReLU(Wx+b)` вводим `a(x)=[h(x),1]` и `theta=[v,b_out]`. Зашумляем именно выходные веса и bias:

$$\widetilde\theta(x)=\theta+\frac{\sigma}{\|a(x)\|_2}\varepsilon,\quad \varepsilon\sim\mathcal N(0,I).$$

Тогда `a(x) @ theta_tilde(x)` имеет распределение `N(f(x), sigma²)` — такое же по дисперсии, как у таргета `N(y(x), sigma²)`. Шум к готовому выходу сети не прибавляется. Постоянная единица в `a(x)` обеспечивает ненулевую норму, даже когда все ReLU неактивны.

**Это локальный, зависящий от x шум параметров**: для разных точек получаются разные нормировки и реализации весов. Это не одна общая случайно зашумлённая сеть для всего интервала.

При σ>0 минимизируем аналитический `KL(N(y, sigma²) || N(f, sigma²))`. Он действительно равен `MSE / (2*sigma²)`: при точном совпадении дисперсий дополнительной регуляризации от интегрированного шума нет. Для вычисления KL сэмплирование не требуется; реальные шумные веса сэмплируются для графика. При σ=0 KL точечных распределений не заменяется выдуманной формулой — последняя итерация явно переключается на MSE.

Сверху — KL до последней итерации, финальный MSE показан отдельной красной точкой; эти значения имеют разный масштаб. В заголовке также указан обычный MSE среднего предсказания. Снизу — средние функции и по одной шумной реализации таргета и выхода. На последней итерации шумные реализации совпадают со своими средними.

In [1]:
from pathlib import Path
import sys

project_root = next(
    p
    for p in (Path.cwd(), *Path.cwd().parents)
    if (p / "pyproject.toml").is_file() and (p / "notebooks" / "_shared").is_dir()
)
shared_parent = str(project_root / "notebooks")
if shared_parent not in sys.path:
    sys.path.insert(0, shared_parent)

import torch
from _shared import Config
from _shared.annealed_weight_noise import NoiseSchedule
from _shared.noise_experiment import AnnealedNoiseExperiment, train_noise_live

torch.set_num_threads(1)

In [2]:
config = Config(
    seed=42,
    group_sizes=(128,),
    steps_per_stage=1600,
    learning_rate=0.01,
    n_train=256,
    n_grid=601,
    x_min=-3.0,
    x_max=3.0,
)
schedule = NoiseSchedule(
    sigma_start=1.0,
    sigma_before_final=0.05,
)
target_fps = 10  # Максимальная частота отрисовки, обучение без пауз.

In [3]:
noise_experiment = AnnealedNoiseExperiment(config, schedule)
await train_noise_live(noise_experiment, target_fps=target_fps)